# **Testing with Concept Activation Vectors (TCAV): Practice**

In [ ]:
# Based on the tutorial https://captum.ai/tutorials/TCAV_Image

**Hi, everyone!**

Continuing the practical application of the theory for concept-based methods, let us move on to the next one — TCAV (Testing with Concept Activation Vectors, [paper](https://arxiv.org/pdf/1711.11279.pdf)). In this practice we will work with it, analysing the solution of a classification task using the GoogleNet model and the already familiar dataset — imagenet.

In this practice you will:
- Work with concepts as an entity of the `captum` library
- Get to know examples from the Broden dataset
- Analyse whether the "stripes" concept influences the model deciding that the image belongs to the zebra class

**Enjoy the coding!**

<a href="https://ibb.co/4KZxwnn"><img src="https://i.ibb.co/jbvF9PP/geranimo-ADUi-P4n-Jwds-unsplash.jpg" alt="geranimo-ADUi-P4n-Jwds-unsplash" border="0"></a>

You have already got acquainted in detail with how T-CAV works in the theory. Let us sum up once again some of its properties (which will come in handy for our practical evaluation).

**By definition:**

- T-CAV $\in [0, ∞]$
- T-CAV $< 0.5$ negative influence of the concept on class $k$
- T-CAV $> 0.5$ positive influence of the concept on class $k$
- T-CAV is computed at the level of a specific layer
- Objects with concepts do NOT take part in training the network

**One of the properties listed is wrong. Find it and select it in the trainer.**

As always, let us first install the necessary dependencies.

In [ ]:
!pip install captum tcav -q

Let us also import everything we will need for the work.

In [ ]:
import numpy as np
import os, glob

import matplotlib.pyplot as plt

from PIL import Image

# ..........torch imports............
import torch
import torchvision

from torch.utils.data import IterableDataset, DataLoader
from torchvision import transforms

#.... Captum imports..................
from captum.attr import LayerGradientXActivation, LayerIntegratedGradients

from captum.concept import TCAV
from captum.concept import Concept

from captum.concept._utils.data_iterator import dataset_to_dataloader, CustomIterableDataset
from captum.concept._utils.common import concepts_to_str


## Loading the images and defining the functions

**1. Images**

To work and to verify the scores, we will need:
- a dataset with specific concepts
- a dataset with random concepts
- a dataset for testing

The data has already been uploaded to GitHub in advance. Download and unpack the data using the following commands one after another:

```
!wget https://github.com/aiedu-courses/all_datasets/blob/5f087ca9bae13513a84520bf0b3a6ea82bbea163/images/tcav.zip?raw=true -O tcav.zip
```

```
!unzip 'tcav.zip'
```

In [ ]:
!wget https://github.com/SadSabrina/explainable_AI_course/raw/refs/heads/main/HW_module12_concept%20based/tcav.zip -O tcav.zip

!unzip 'tcav.zip'

After unpacking, a `tcav` folder should appear. Check that all the images are there in the `concepts` and `data` folders.

- `concepts` — we will work with five concepts, three specific ones — `striped`, `zigzagged` and `dotted` — and two random ones.

 The concepts `striped`, `zigzagged` and `dotted` are taken from the [broden](https://netdissect.csail.mit.edu/broden1_224) dataset.

 The concepts `random0`, `random1` and `random2` are taken from the imagenet dataset.

- `data` — we will test and evaluate the concepts on the `data` dataset, which consists of zebras.

An alternative way of obtaining the data is also described in the [tutorial](https://github.com/tensorflow/tcav/tree/master/tcav/tcav_examples/image_models/imagenet) from tensorflow.

**2. Functions**

Next, to process the data we will need:
- functions for loading
- a function for preprocessing
- a dataset that can be iterated over

1. A function for preprocessing the data.

In [ ]:
# A function for preprocessing the data
def transform(img):

    return transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            ),
        ]
    )(img)

2. Data loading functions.

- `load_image_tensors` — for loading images as tensors from a specific path

**Analyse the code of the function. What is returned if the flag `transform=False`?**

In [ ]:
# Functions for loading the data
general_path = '/content/tcav' # specify the path to the tcav folder in the form '.../.../tcav'


def load_image_tensors(class_name, root_path=general_path + '/data/', transform=True):

    path = os.path.join(root_path, class_name) # specify the path to the directory with the images
    filenames = glob.glob(path + '/*.jpg') # read all the images with the .jpg pattern

    tensors = []
    for filename in filenames:                    # convert the pictures into tensors
        img = Image.open(filename).convert('RGB')
        tensors.append(transform(img) if transform else img)

    return tensors

- `get_tensor_from_filename` — a function for loading an image from the specified path as a tensor.
- `assemble_concept` — a function for reading concepts using the path to the directory they are in, and creating a special object.

**What does the `assemble_concept` function return?**

In [ ]:
def get_tensor_from_filename(filename):

    img = Image.open(filename).convert("RGB") # a simple function for processing an image
    return transform(img)

def assemble_concept(name, id, concepts_path=general_path + "/concepts"): # A function for creating a concept object

    concept_path = os.path.join(concepts_path, name) + "/" # get the path to the folder with the concepts
    dataset = CustomIterableDataset(get_tensor_from_filename, concept_path)  # get tensors from every object in the folder
    concept_iter = dataset_to_dataloader(dataset)

    return Concept(id=id, name=name, data_iter=concept_iter)

Let us prepare an object based on each concept.

In [ ]:
concepts_path = general_path + '/concepts'

stripes_concept = assemble_concept("striped", 0, concepts_path=concepts_path)
zigzagged_concept = assemble_concept("zigzagged", 1, concepts_path=concepts_path)
dotted_concept = assemble_concept("dotted", 2, concepts_path=concepts_path)


random_0_concept = assemble_concept("random500_0", 3, concepts_path=concepts_path)
random_1_concept = assemble_concept("random500_1", 4, concepts_path=concepts_path)

Let us make use of the helper functions and visualise the loaded concepts.

In [ ]:
n_figs = 5
n_concepts = 5

fig, axs = plt.subplots(n_concepts, n_figs + 1, figsize = (20, 20))

for c, concept in enumerate([stripes_concept, zigzagged_concept, dotted_concept, random_0_concept, random_1_concept]):

    concept_path = os.path.join(concepts_path, concept.name) + "/"
    img_files = glob.glob(concept_path + '*.jpg')

    for i, img_file in enumerate(img_files[:n_figs + 1]):
        if os.path.isfile(img_file):
            if i == 0:
                axs[c, i].text(1.0, 0.5, str(concept.name), ha='right', va='center', family='sans-serif', size=20)
            else:
                img = plt.imread(img_file)
                axs[c, i].imshow(img)

            axs[c, i].axis('off')


## The model

As has already been described, we will work with the GoogleNet model. A description of the model can be found in the [original paper](https://arxiv.org/pdf/1409.4842v1).

In [ ]:
model = torchvision.models.googlenet(pretrained=True)
model = model.eval()

**Examine the architecture of the model. Select the correct statement in the trainer.**

# Computing the T-CAV scores.

Let us recall the mathematical definition of T-CAV:
$$T_{-}CAV_{c, k, l}=\frac{|x \in X_k: S_{c, k, l}(x) > 0|}{|X_k|}$$

where
- the symbols $|\cdot|$ denote the cardinality of a set (in our case the cardinality of the set is exactly equal to the number of objects in it)
- the scores $S_{c, k, l}(x)$ reflect the directional derivative towards the presence of concept "c".

In the theory we noted that this looks like the task of classifying objects into those sensitive to the concept and those not sensitive to it. And that is indeed so!

Let us figure out how to set up T-CAV. You need to specify:
- model — the model TCAV will be applied to,
- layers — a list with the names of the layers on which the changes of the activations will be analysed
- classifier — the classifier that will split the objects into sensitive and not sensitive ones (captum makes it possible to define your own classifiers, which are implemented analogously to an extended sklearn.linear_model. You can find an example [here](https://captum.ai/api/_modules/captum/concept/_utils/classifier.html). By default captum uses [sklearn.linear_model.SGDClassifier](https://scikit--learn-org.translate.goog/stable/modules/generated/sklearn.linear_model.SGDClassifier.html?_x_tr_sl=en&_x_tr_tl=ru&_x_tr_hl=ru&_x_tr_pto=sc), you can omit this parameter)
- layer_attr_method — the method according to which the attributions will be computed
- save_path — also an optional parameter, if you want to save the Class Activation Vectors (CAVs) and the Activation Vectors (AVs) themselves.

We will use TCAV with the classic captum classifier on 3 layers — `['inception4c', 'inception4d', 'inception4e']`. As the attribution method we will use Integrated Gradients at the layer level.

In [ ]:
layers=['inception4c', 'inception4d', 'inception4e']

mytcav = TCAV(model=model,
              layers=layers,
              layer_attr_method = LayerIntegratedGradients(
                model, None, multiply_by_inputs=False))


**Computing TCAV.**

**Let us formulate the task:** evaluate the importance of the "stripes" concept for predicting the zebra class.

You will agree that the importance score for this class alone will not be enough for us — we need to make a comparison. To do this, let us make two experimental sets: ["striped", "random_0"] and ["striped", "random_1"].

From the point of view of the theory, our classifier will build a hyperplane that separates the concepts of our sets from each other.

In [ ]:
experimental_set_rand = [[stripes_concept, random_0_concept], [stripes_concept, random_1_concept]]


Let us load the data on which we will look for stripes. And we will look for them on one of the most striped creatures in the world — on zebras!

The data lies at the path `/tcav/data/zebra`. If you unpacked everything correctly and specified the paths above correctly, just run the code cell below. If there are errors, check that the path is correct.

In [ ]:
# Loading the zebras
zebra_imgs = load_image_tensors('zebra', transform=False)

In [ ]:
print(f'We have as many as {len(zebra_imgs)} pictures of zebras!')

Let us visualise some of them.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize = (25, 5))
axs[0].imshow(zebra_imgs[34])
axs[1].imshow(zebra_imgs[15])
axs[2].imshow(zebra_imgs[7])

axs[0].axis('off')
axs[1].axis('off')
axs[2].axis('off')


plt.show()

Let us convert the zebras into tensors.

In [ ]:
zebra_tensors = torch.stack([transform(img) for img in zebra_imgs])


Before going further, let us make sure that the model really does "see" zebras on all the images. Check this by running the prediction process and extracting the predicted class.

**How many pictures are classified by the model as a zebra?**

In [ ]:
# Your code here

And, finally, let us launch the interpretation process!

In [ ]:
# zebra class index
zebra_ind = 340


tcav_scores_w_random = mytcav.interpret(inputs=zebra_tensors,
                                        experimental_sets=experimental_set_rand,
                                        target=zebra_ind,
                                        n_steps=5,
                                       )
tcav_scores_w_random

In [ ]:
experimental_set_rand

So, we have obtained a dictionary of TCAV scores. Let us figure out what each of them means:
- sign count — computed by the formula $\frac{| TCAV > 0 |}{|TCAV|}$

- magnitude score  — computed as $\frac{SUM(ABS(TCAV * (TCAV > 0)))}{SUM(ABS(TCAV))}$

The first score corresponds to the first concept in `experimental_set_rand`, the second one to the second. As you may notice, in this implementation only the sign count scores look correct (magnitude by definition cannot be less than 0) **(a note from the instructor)**.

**Suppose we are comparing TCAV between two sets of concepts and the sign count of one of them equals $0.8573$. What does the sign count of the second one equal? Do not round the answer.**

Let us visualise the obtained `sign_count` values by layer in the first and the second experiment.

In [ ]:
# Let us extract the data for drawing on the plot

# The first experiment

first_experiment_concepts = [experimental_set_rand[0][0].name, experimental_set_rand[0][1].name] # concepts of the first experiment
first_experiment_layers = list(tcav_scores_w_random['0-3'].keys()) # layers of the first experiment
first_experiment_scores = [tcav_scores_w_random['0-3'][layer]['sign_count'].numpy() for layer in first_experiment_layers]

In [ ]:
# The second experiment

second_experiment_concepts = [experimental_set_rand[1][0].name, experimental_set_rand[1][1].name] # concepts of the second experiment
second_experiment_layers = list(tcav_scores_w_random['0-4'].keys()) # layers of the first experiment
second_experiment_scores = [tcav_scores_w_random['0-4'][layer]['sign_count'].numpy() for layer in second_experiment_layers] # scores of the second experiment

In [ ]:
fig, ax = plt.subplots(2, 3, figsize = (18, 10))

for i in range(0, len(first_experiment_layers)):

  ax[0, i].bar(first_experiment_concepts, first_experiment_scores[i], color=['mediumspringgreen', 'purple'])
  ax[0, i].set_title(first_experiment_layers[i])
  ax[0, i].bar_label(ax[0, i].containers[0])

for i in range(0, len(second_experiment_layers)):

  ax[1, i].bar(second_experiment_concepts, second_experiment_scores[i], color=['mediumspringgreen', 'lightblue'])
  ax[1, i].set_title(second_experiment_layers[i])
  ax[1, i].bar_label(ax[1, i].containers[0])


fig.suptitle('TCAV of two different concepts in first and second\n experiments', fontsize=15);

From the plots above you can note that the images which are predicted by the model as `zebra` are very sensitive to the `striped` concept, unlike the random dataset. However, since the random pictures are drawn uniformly from the imagenet dataset — they do not represent any specific concept, so they can be considered noise.

Now let us formulate an experiment with specific concepts — let us put dots and zigzags next to our stripes.

In [ ]:
experimental_set_zig_dot = [[stripes_concept, zigzagged_concept, dotted_concept]] # The experimental set

In [ ]:
tcav_scores_w_zig_dot = mytcav.interpret(inputs=zebra_tensors,
                                         experimental_sets=experimental_set_zig_dot, # Launching the interpretation process for the new concepts
                                         target=zebra_ind,
                                         n_steps=5)

In [ ]:
tcav_scores_w_zig_dot

In [ ]:
# Let us extract the data of the third experiment in the same way — for drawing on the plot

experiment_concepts = [experimental_set_zig_dot[0][0].name, experimental_set_zig_dot[0][1].name, experimental_set_zig_dot[0][2].name]
experiment_layers = list(tcav_scores_w_zig_dot['0-1-2'].keys())
experiment_scores = [tcav_scores_w_zig_dot['0-1-2'][layer]['sign_count'].numpy() for layer in experiment_layers]

experiment_scores

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (18, 6))

for i in range(0, len(experiment_layers)):

  ax[i].bar(experiment_concepts, experiment_scores[i], color=['mediumspringgreen', 'purple', 'lightblue'])
  ax[i].set_title(experiment_layers[i])
  ax[i].bar_label(ax[i].containers[0])

fig.suptitle('TCAV with three meaningful concepts', fontsize=15);

1. As you can see, the sum of the values somewhat exceeds one — this is normal, since we are analysing the shift towards the presence of the concept for each of the three concepts.
2. Similarly to the previous one, in this experiment we also observe that the `striped` concept has very high TCAV scores in all three layers compared with `zigzagged` and `dotted`. This means that the `striped` concept is rather an essential concept in predicting `zebra`.

**That is all! A small but, we hope, useful practice.**

**Thank you for your work and attention!** 😊